# Lesson 1d — Autograd from scratch (runnable)

We've used `loss.backward()` everywhere. But what does it actually DO?

This notebook builds a tiny autograd engine — `micrograd`-style, following Karpathy's lead. ~100 lines of Python that does what PyTorch does (for scalars), and matches PyTorch's gradients exactly.

By the end you'll know:
- Why each tensor needs a `.grad` field and a "backward function"
- What "the chain rule" looks like in code
- That `.backward()` is just a topological walk of the computation graph

Runnable version of [`01d_autograd_from_scratch.py`](../01d_autograd_from_scratch.py).

## Step 1 — The `Value` class

Each `Value` holds:
- `.data` — the numerical value
- `.grad` — the gradient (set by `.backward()`)
- `._prev` — Values it was built from
- `._backward` — a function that applies the chain rule

In [ ]:
class Value:
    def __init__(self, data, _children=(), _op=""):
        self.data = data
        self.grad = 0.0                     # Will be set by .backward().
        self._prev = set(_children)         # Parents in the computation graph.
        self._op = _op                      # Label for debugging.
        self._backward = lambda: None       # No-op for leaf nodes.

    def __repr__(self):
        return f"Value(data={self.data:.4f}, grad={self.grad:.4f})"

    # ADDITION: if c = a + b, then dc/da = 1 and dc/db = 1.
    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), "+")
        def _backward():
            self.grad  += out.grad          # Chain rule: 1 * out.grad
            other.grad += out.grad
        out._backward = _backward
        return out

    # MULTIPLICATION: if c = a * b, then dc/da = b and dc/db = a.
    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), "*")
        def _backward():
            self.grad  += other.data * out.grad
            other.grad += self.data  * out.grad
        out._backward = _backward
        return out

    # POWER (constant exponent): if c = a**n, then dc/da = n * a**(n-1).
    def __pow__(self, n):
        out = Value(self.data ** n, (self,), f"**{n}")
        def _backward():
            self.grad += (n * self.data ** (n - 1)) * out.grad
        out._backward = _backward
        return out

    def __neg__(self):     return self * -1
    def __sub__(self, o):  return self + (-o)
    def __radd__(self, o): return self + o
    def __rmul__(self, o): return self * o

    # THE BIG ONE: .backward()
    def backward(self):
        topo = []                           # Topologically sorted graph.
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        build_topo(self)

        self.grad = 1.0                     # Seed: dL/dL = 1.
        for v in reversed(topo):            # Walk in REVERSE topological order.
            v._backward()                   # Apply chain rule at each node.

## Step 2 — Sanity check

Take `f(x) = 3x² + 2x + 5`. We know `df/dx = 6x + 2`. At `x = 4`, gradient should be `26`.

In [ ]:
x = Value(4.0)
f = Value(3) * x ** 2 + Value(2) * x + Value(5)
f.backward()

print(f"f(4) = 3*16 + 2*4 + 5 = {f.data}    (expected 61)")
print(f"df/dx at x=4 = 6*4 + 2 = {x.grad}    (expected 26)")
assert abs(x.grad - 26) < 1e-6
print("✓ Correct.")

## Step 3 — Train Lesson 1's linear regression with OUR autograd

Linear regression on `y = 2x + 1`. NO PyTorch autograd — just our `Value` class.

In [ ]:
xs = [1.0, 2.0, 3.0, 4.0, 5.0]
ys = [2 * x + 1 for x in xs]

w = Value(0.0)
b = Value(0.0)
lr = 0.05

for step in range(101):
    # Forward
    loss = Value(0.0)
    for xv, yv in zip(xs, ys):
        pred = w * Value(xv) + b
        err  = pred - Value(yv)
        loss = loss + err ** 2
    loss = loss * Value(1.0 / len(xs))      # Mean.

    # Zero gradients manually (no opt.zero_grad helper here)
    w.grad = 0.0
    b.grad = 0.0

    loss.backward()                          # Compute gradients.

    # Step (manual SGD)
    w.data -= lr * w.grad
    b.data -= lr * b.grad

    if step % 20 == 0:
        print(f"  step {step:3d}   w={w.data:.3f}   b={b.data:.3f}   loss={loss.data:.4f}")

print(f"\nFinal: w = {w.data:.3f}, b = {b.data:.3f}    (target: w=2, b=1)")

## Step 4 — Compare to PyTorch's autograd at step 0

Recreate Lesson 1c's hand-computed step-0 gradients in our autograd. Confirm we match PyTorch.

In [ ]:
import torch

# Ours
w = Value(0.0)
b = Value(0.0)
loss = Value(0.0)
for xv, yv in zip(xs, ys):
    err = w * Value(xv) + b - Value(yv)
    loss = loss + err ** 2
loss = loss * Value(1.0 / len(xs))
w.grad = 0; b.grad = 0
loss.backward()
print(f"  Ours:     loss = {loss.data:.2f},  grad_w = {w.grad:.2f},  grad_b = {b.grad:.2f}")

# PyTorch
wt = torch.tensor(0.0, requires_grad=True)
bt = torch.tensor(0.0, requires_grad=True)
xt = torch.tensor(xs)
yt = torch.tensor(ys)
lt = ((wt * xt + bt - yt) ** 2).mean()
lt.backward()
print(f"  PyTorch:  loss = {lt.item():.2f},  grad_w = {wt.grad.item():.2f},  grad_b = {bt.grad.item():.2f}")
print()
print("Same loss, same gradients. Our 100-line autograd does what PyTorch does.")

## Why this matters

`loss.backward()` in PyTorch is NOT magic. Every operation has a registered "backward function" that pushes gradients back via the chain rule. `.backward()` walks the computation graph from loss back to the leaves (parameters), accumulating gradients.

Every Transformer, every LLM is trained using exactly this mechanism. **Chain rule + topo walk = the entire foundation of deep learning.**

## Things to try

1. Add `.tanh()` to `Value`. `dy/dx = 1 - y²`.
2. Add `.exp()` and `.relu()`. Now you can build MLPs entirely in our autograd.
3. (Stretch) Generalise `Value` to hold tensors. Be careful with broadcasting in backward.